# SDG Hub — Seed Data Semi-automático (Notebook de estudos)

Este notebook cria um **pequeno** `seed_data.jsonl` no formato esperado pelo **SDG Hub** e, em seguida, roda um flow de geração de dados a partir desse seed.

> Objetivo: manter o **padrão do SDG Hub** (seed JSONL → `FlowRegistry` → `Flow.from_yaml()` → `flow.run()`), mas com **seed semi-automático**:  
> - `document_outline` gerado por LLM  
> - `icl_document` selecionado automaticamente (heurísticas simples)  
> - `icl_query_1..3` geradas por LLM e parseadas em colunas  
>
> Referências (padrão base):
> - `document_pre_processing.ipynb` (seed manual)  
> - `knowledge_generation.ipynb` (consumo do seed e execução de flows)

---

## Pré-requisitos

1. SDG Hub instalado (recomendado): `pip install sdg-hub[examples]`
2. `.env` com configuração do provider/modelo (ou variáveis de ambiente equivalentes)



In [ ]:
# Step 0: Instalação (se necessário)
# Observação: em ambientes já preparados, você pode pular esta célula.

%pip install -U sdg-hub[examples] python-dotenv datasets markdown-it-py nest_asyncio


In [ ]:
# Step 0.1: Imports e .env
from dotenv import load_dotenv
import os
from pathlib import Path

load_dotenv()


In [ ]:
# Required to run the flow with async mode (padrão do SDG Hub)
import nest_asyncio
nest_asyncio.apply()


## Step 1: Document Processing Pipeline (parse -> markdown)

Este passo replica o notebook oficial: usa o parser do InstructLab (Docling v2) para transformar arquivos em `.md`.

- Coloque documentos (PDF/DOCX/etc.) em `document_collection/`
- O comando abaixo gera `.md` na mesma pasta

> Se você **já** tem `.md`, pode pular e ir para o Step 2.


In [ ]:
# Step 1: Document Processing Pipeline
data_dir = "document_collection/"

# Parser do exemplo oficial do sdg_hub (instructlab/docling)
# Se o arquivo ../docparser_v2.py não existir no seu checkout, ajuste o path conforme seu repo.
!python ../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../docling_v2_config.yaml


## Step 2: Carregar um documento Markdown processado

Para estudo, vamos trabalhar com **um** `.md` (o primeiro encontrado).


In [ ]:
import glob

md_files = sorted(glob.glob(f"{data_dir}/*.md"))
if not md_files:
    raise FileNotFoundError(f"Nenhum .md encontrado em {data_dir}. Rode o Step 1 ou coloque um .md lá.")

md_path = md_files[0]
print("Using:", md_path)

with open(md_path, "r", encoding="utf-8") as f:
    text = f.read()

print("Chars:", len(text))
print(text[:800])


## Step 3: Chunking (padrão compatível com o notebook do SDG Hub)

Vamos reutilizar a mesma ideia do exemplo: chunking em nível de blocos markdown e overlap.


In [ ]:
from markdown_it import MarkdownIt
from typing import List
import re

def chunk_markdown(text: str, max_tokens: int = 800, overlap: int = 120) -> List[str]:
    """Chunking simples em nível de blocos (headings/parágrafos/listas etc.) + overlap em palavras."""
    md = MarkdownIt()
    tokens = md.parse(text)

    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
            buf = []
        elif tok.content:
            buf.append(tok.content)

    if buf:
        blocks.append("\n".join(buf).strip())

    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                chunks.append(" ".join(current_words))
                current_words = current_words[-overlap:] if overlap > 0 else []
    if current_words:
        chunks.append(" ".join(current_words))
    return [c.strip() for c in chunks if c.strip()]

chunks = chunk_markdown(text, max_tokens=int(os.getenv("CHUNK_MAX_TOKENS", "800")), overlap=int(os.getenv("CHUNK_OVERLAP", "120")))
print("chunks:", len(chunks))
print(chunks[0][:600])


## Step 4: Seed semi-automático (outline + icl_document + icl_queries)

No seed “manual” do exemplo, você preenche na mão:
- `document_outline`
- `icl_document`
- `icl_query_1..3`
- `domain`

Aqui vamos:
1) Selecionar automaticamente um `icl_document` (heurísticas simples)
2) Gerar `document_outline` com LLM (via bloco SDG Hub)
3) Gerar 3 queries com LLM (via bloco SDG Hub) e parsear para colunas

### 4.1 Seleção automática do `icl_document`
Heurística (boa o bastante para começar pequeno):
- descartar chunks muito curtos
- penalizar boilerplate típico (copyright, “table of contents”, etc.)
- escolher o chunk com melhor score



In [ ]:
def score_chunk(c: str) -> float:
    # Penaliza boilerplate comum
    boiler = [
        "table of contents", "copyright", "all rights reserved",
        "disclaimer", "confidential", "license", "spdx"
    ]
    c_low = c.lower()
    penalty = sum(1 for b in boiler if b in c_low)

    # Recompensa tamanho “útil”
    n = len(c.split())
    if n < 120:
        return -999

    # Penaliza excesso de linhas muito curtas (toc/lista de links)
    short_lines = sum(1 for line in c.splitlines() if 0 < len(line.strip()) < 20)
    return n - (penalty * 200) - (short_lines * 5)

best_idx, best_score = max(enumerate(chunks), key=lambda t: score_chunk(t[1]))
icl_document = chunks[best_idx]
print("Selected icl_document idx:", best_idx, "score:", best_score)
print(icl_document[:800])


### 4.2 Construir um dataset seed (padrão SDG Hub)

O contrato do seed é o mesmo do notebook oficial: a coluna base é `document` (chunk) e adicionamos campos ICL.


In [ ]:
import datasets
seed_data = datasets.Dataset.from_dict({"document": chunks})
print(seed_data)


### 4.3 Geração do `document_outline` e `icl_queries` usando **Blocks** do SDG Hub

Aqui está o ponto “100% SDG Hub”: usamos **blocks** (por exemplo, `LLMChatBlock`) para gerar campos e manter a execução/validação no padrão do framework.

> Observação: você pode trocar o provider/modelo via `flow.set_model_config()` ou variáveis no `.env`.


In [ ]:
# Third Party
import pandas as pd

# First Party (SDG Hub)
from sdg_hub.core.blocks import LLMChatBlock
try:
    # Nem todas as versões expõem este bloco, então mantemos opcional.
    from sdg_hub.core.blocks import JSONStructureBlock
    HAS_JSON_BLOCK = True
except Exception:
    HAS_JSON_BLOCK = False

from sdg_hub import Flow


In [ ]:
# Helper: configurar modelo nos blocks via Flow (mesmo padrão do knowledge_generation.ipynb)
def set_model_config(flow_object):
    model_provider = os.getenv("MODEL_PROVIDER", "hosted_vllm")
    print(f"Using model provider: {model_provider}")

    if model_provider == "hosted_vllm":
        vllm_model = os.getenv("VLLM_MODEL", "hosted_vllm/meta-llama/Llama-3.3-70B-Instruct")
        vllm_api_base = os.getenv("VLLM_API_BASE", "http://localhost:8000/v1")
        vllm_api_key = os.getenv("VLLM_API_KEY", "EMPTY")
        enable_reasoning = os.getenv("ENABLE_REASONING", "false").lower() in ("1","true","yes")
        flow_object.set_model_config(model=vllm_model, api_base=vllm_api_base, api_key=vllm_api_key, enable_reasoning=enable_reasoning)

    elif model_provider == "openai":
        openai_api_key = os.getenv("OPENAI_API_KEY")
        openai_model = os.getenv("OPENAI_MODEL", "openai/gpt-4")
        flow_object.set_model_config(model=openai_model, api_key=openai_api_key)

    elif model_provider == "openrouter":
        openai_api_key = os.getenv("OPENAI_API_KEY")
        openai_model = os.getenv("OPENAI_MODEL", "openai/gpt-4")
        flow_object.set_model_config(model=openai_model, api_key=openai_api_key, api_base="https://openrouter.ai/api/v1")

    elif model_provider == "ollama":
        ollama_model = os.getenv("OLLAMA_MODEL", "ollama/gemma2")
        ollama_api_base = os.getenv("OLLAMA_API_BASE", "http://localhost:11434")
        flow_object.set_model_config(model=ollama_model, api_base=ollama_api_base)

    elif model_provider == "maas":
        maas_model = os.getenv("MAAS_MODEL")
        maas_api_base = os.getenv("MAAS_API_BASE")
        maas_api_key = os.getenv("MAAS_API_KEY")
        flow_object.set_model_config(model=maas_model, api_base=maas_api_base, api_key=maas_api_key)

    return flow_object


#### 4.3.1 Dataset “ICL input row” (1 linha)

Vamos criar uma linha com o `icl_document` selecionado para pedir ao LLM:
- um outline curto
- um domínio (taxonomia simples)
- 3 perguntas (Q1 factual, Q2 procedural, Q3 edge case/comparação)

Depois distribuímos esses campos para todos os chunks (mesmo padrão do exemplo manual).


In [ ]:
icl_df = pd.DataFrame([{
    "icl_document": icl_document
}])

icl_df.head()


In [ ]:
# Bloco 1: gerar document_outline + domain (texto)
outline_block = LLMChatBlock(
    block_name="gen_outline",
    input_cols=["icl_document"],
    output_cols=["document_outline_domain_raw"],
    prompt_template=(
        "You are generating seed metadata for knowledge tuning.
"
        "Given the document excerpt below, produce:
"
        "1) A concise 1-2 sentence document outline (high-level)
"
        "2) A single domain label (choose ONE): Finance, Legal, IT, HR, Security, Operations, Healthcare, Education, Other

"
        "Return exactly in this format:
"
        "OUTLINE: <text>
"
        "DOMAIN: <label>

"
        "EXCERPT:
{icl_document}
"
    ),
)

# Bloco 2: gerar 3 perguntas (pedimos JSON para facilitar parsing)
questions_block = LLMChatBlock(
    block_name="gen_icl_questions",
    input_cols=["icl_document"],
    output_cols=["icl_questions_json"],
    prompt_template=(
        "You are creating 3 high-quality, answerable questions for knowledge tuning.
"
        "The questions MUST be answerable using ONLY the excerpt.
"
        "Generate:
"
        "- Q1: factual/definition
"
        "- Q2: procedural/how-to or requirements
"
        "- Q3: edge case / restriction / comparison

"
        "Return STRICT JSON with keys: icl_query_1, icl_query_2, icl_query_3.

"
        "EXCERPT:
{icl_document}
"
    ),
)


In [ ]:
# Executar os blocks em modo "flow-like": instanciamos um Flow simples e executamos blocks sequencialmente.
# (Mantém o estilo SDG Hub: blocks com validação + execução padronizada.)
tmp_flow = Flow(metadata={"name":"Seed Semi-auto (study)","description":"Generate outline/domain + icl queries"})
tmp_flow.blocks = [outline_block, questions_block]
tmp_flow = set_model_config(tmp_flow)

# Rodar no dataframe (sdg_hub >=0.8 usa pandas internamente)
result_df = tmp_flow.run(icl_df, max_concurrency=int(os.getenv("MAX_CONCURRENCY","4")))
result_df


In [ ]:
# Parse do OUTLINE/DOMAIN
raw = result_df.loc[0, "document_outline_domain_raw"]
outline_match = re.search(r"OUTLINE:\s*(.+)", raw)
domain_match = re.search(r"DOMAIN:\s*(.+)", raw)

document_outline = (outline_match.group(1).strip() if outline_match else raw.strip())
domain = (domain_match.group(1).strip() if domain_match else "Other")

# Parse do JSON de perguntas
import json
q_raw = result_df.loc[0, "icl_questions_json"]
try:
    q_obj = json.loads(q_raw)
except Exception:
    # fallback: tenta extrair linhas "icl_query_1:" etc.
    q_obj = {}
    for k in ["icl_query_1","icl_query_2","icl_query_3"]:
        m = re.search(rf"{k}\s*[:=]\s*(.+)", q_raw, flags=re.IGNORECASE)
        if m:
            q_obj[k] = m.group(1).strip()

icl_query_1 = q_obj.get("icl_query_1","").strip()
icl_query_2 = q_obj.get("icl_query_2","").strip()
icl_query_3 = q_obj.get("icl_query_3","").strip()

print("document_outline:", document_outline)
print("domain:", domain)
print("icl_query_1:", icl_query_1)
print("icl_query_2:", icl_query_2)
print("icl_query_3:", icl_query_3)


### 4.4 Montar o seed final (map para todas as linhas) e salvar JSONL

Mesma lógica do exemplo oficial: todos os chunks recebem os mesmos campos ICL (para um estudo pequeno).


In [ ]:
icl_fields = {
    "document_outline": document_outline,
    "icl_document": icl_document,
    "icl_query_1": icl_query_1,
    "icl_query_2": icl_query_2,
    "icl_query_3": icl_query_3,
    "domain": domain,
}

seed_data_final = seed_data.map(lambda _: icl_fields)
seed_path = os.getenv("SEED_DATA_PATH", "seed_data.jsonl")
seed_data_final.to_json(seed_path, orient="records", lines=True)
print("Saved:", seed_path)
print(seed_data_final[0])


## Step 5: Rodar um flow de geração do SDG Hub em cima do seed

Aqui seguimos o padrão do `knowledge_generation.ipynb`:
- `FlowRegistry.discover_flows()`
- escolher um flow (ex.: **Document Based Knowledge Tuning Dataset Generation Flow**)
- `Flow.from_yaml(flow_path)`
- `set_model_config(flow)`
- `flow.run(seed_df, ...)`
- salvar JSONL do output

> Para estudo, vamos:
> - subsample (ex.: 5 chunks)
> - ajustar runtime params se necessário


In [ ]:
from sdg_hub import FlowRegistry
from datasets import load_dataset

FlowRegistry.discover_flows()
print("Total flows:", len(FlowRegistry.list_flows()))


In [ ]:
# Carregar seed e subsample pequeno
seed_ds = load_dataset("json", data_files=seed_path, split="train")
subsample = int(os.getenv("SEED_DATA_SUBSAMPLE", "5"))
if subsample > 0:
    seed_ds = seed_ds.select(range(min(subsample, len(seed_ds))))

seed_df = seed_ds.to_pandas()
seed_df.head()


In [ ]:
# Escolher um flow pronto (document-based é simples e rápido)
flow_name = "Document Based Knowledge Tuning Dataset Generation Flow"

flow_path = FlowRegistry.get_flow_path(flow_name)
print("Flow path:", flow_path)

gen_flow = Flow.from_yaml(flow_path)
gen_flow = set_model_config(gen_flow)


In [ ]:
# Rodar o flow com poucos docs
# runtime_params: você pode ajustar parâmetros específicos do flow (ex.: temperatura, max_tokens, etc.)
runtime_params = {
    # Exemplo (se existir no flow): "question_generation": {"temperature": 0.7}
}

generated_df = gen_flow.run(
    seed_df,
    runtime_params=runtime_params,
    max_concurrency=int(os.getenv("MAX_CONCURRENCY","4")),
)

generated_df.head()


In [ ]:
# Salvar resultado (pequeno dataset)
out_path = os.getenv("OUTPUT_DATA_PATH", "generated_knowledge.jsonl")
Path(out_path).parent.mkdir(parents=True, exist_ok=True)

generated_df.to_json(out_path, orient="records", lines=True)
print("Saved:", out_path)
print("Rows:", len(generated_df))


## Observações finais

- Se você quiser melhorar a robustez “semi-automática”, os próximos upgrades naturais são:
  1) gerar 2–3 candidatos a `icl_document` e pedir para o LLM escolher o melhor
  2) rodar um bloco “judge” rápido para checar se cada pergunta é respondível pelo trecho
  3) amostrar e revisar manualmente uma fração dos seeds

Mas, para estudos, este notebook já mantém:
- **contrato de seed data** compatível com o SDG Hub (`seed_data.jsonl`)
- **execução de flows via FlowRegistry** (padrão oficial)
- **uso de blocks SDG Hub** para automatizar outline/perguntas (sem fugir do framework)
